### Loading the data


In [4]:
import numpy as np
import anndata as ad
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings
import random
import muon as mu


from scipy.sparse import issparse
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda"

# Part 1: RNA Embeddings

In [6]:
# Loading the data

mdata = mu.read("/home/lemanic-g04/GSE194315_raw_mu.h5mu")
rna = mdata["rna"]
protein = mdata["protein"]

# Align cells present in both modalities
common_cells = rna.obs_names.intersection(protein.obs_names)
rna = rna[common_cells].copy()
protein = protein[common_cells].copy()

print(f"RNA:     {rna.shape}")
print(f"Protein: {protein.shape}")
print(f"Common cells: {len(common_cells)}")


I0000 00:00:1776349872.248370     951 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776349873.835025     951 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776349878.594898     951 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


RNA:     (180794, 33694)
Protein: (180794, 273)
Common cells: 180794


In [ ]:
# Creating the RNA embeddings

adata_rna = rna.copy()

# Use raw counts if available
if "counts" in adata_rna.layers:
    adata_rna.X = adata_rna.layers["counts"].copy()

sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)
sc.pp.highly_variable_genes(adata_rna, n_top_genes=2000)
adata_rna = adata_rna[:, adata_rna.var["highly_variable"]].copy()
sc.pp.scale(adata_rna, max_value=10)

# Use full 2000 HVG expression vectors instead of PCA
from scipy.sparse import issparse as issparse_check
rna_matrix = adata_rna.X
if issparse_check(rna_matrix):
    rna_matrix = rna_matrix.toarray()
rna_encodings = np.array(rna_matrix, dtype=np.float32)

D_RNA_ENC = rna_encodings.shape[1]  # 2000
print(f"RNA encodings (full HVG): {rna_encodings.shape}")


# Part 2: Protein Embeddings

In [12]:
# Function to clean protein names (removing the - and the /)

def clean_protein_name(name):
    # "CD107a|LAMP-1" -> "CD107a"
    name = name.split("|")[0]
    # "CD14-1" -> "CD14", but keep "LOX-1"
    parts = name.rsplit("-", 1)
    if len(parts) == 2 and parts[1].isdigit():
        name = parts[0]
    return name.strip()

protein_names_raw = list(protein.var_names)
protein_names = [clean_protein_name(p) for p in protein_names_raw]

#print("Changed names (before -> after):")
#for raw, clean in zip(protein_names_raw, protein_names):
#    if raw != clean:
#        print(f"  {raw:<45} -> {clean}")

# Over 6 characters
long_names = [(i, p) for i, p in enumerate(protein_names) if len(p) > 6]
#print(f"\nProteins with name > 6 chars ({len(long_names)}):")
#for i, p in long_names:
#    print(f"  [{i:>3}] {p}")

# Duplicates
from collections import Counter
counts = Counter(protein_names)
duplicates = {name: cnt for name, cnt in counts.items() if cnt > 1}
#print(f"\nDuplicate names ({len(duplicates)}):")
#if duplicates:
#    for name, cnt in sorted(duplicates.items(), key=lambda x: -x[1]):
#        indices = [i for i, p in enumerate(protein_names) if p == name]
#        raw_originals = [protein_names_raw[i] for i in indices]
#        print(f"  '{name}' appears {cnt}x — from: {raw_originals}")
#else:
#    print("  None")

#print(f"\nTotal proteins: {len(protein_names)}")
#print(f"Unique proteins: {len(set(protein_names))}")


## a) Caption Generation

In [6]:

def generate_caption(binary_profile, protein_names, cell_type=None, n_proteins=None):
    """Generate a natural language caption describing a cell's protein expression.
    
    Randomly samples a subset of proteins and describes which are expressed/not expressed.
    This stochastic captioning acts as data augmentation during training.
    """
    expressed = [p for p, b in zip(protein_names, binary_profile) if b == 1]
    not_expressed = [p for p, b in zip(protein_names, binary_profile) if b == 0]
    
    # Randomly choose how many proteins to mention (between 3 and 15)
    if n_proteins is None:
        n_proteins = random.randint(3, min(15, len(protein_names)))
    
    # Sample from expressed and not-expressed
    n_pos = min(random.randint(1, n_proteins - 1), len(expressed))
    n_neg = min(n_proteins - n_pos, len(not_expressed))
    
    sampled_pos = random.sample(expressed, n_pos) if len(expressed) >= n_pos else expressed
    sampled_neg = random.sample(not_expressed, n_neg) if len(not_expressed) >= n_neg else not_expressed
    
    # Build caption
    parts = []
    if cell_type:
        parts.append(f"This {cell_type}")
    else:
        parts.append("This cell")
    
    if sampled_pos:
        parts.append(f"expresses {', '.join(sampled_pos)}")
    if sampled_neg:
        parts.append(f"does not express {', '.join(sampled_neg)}")
    
    caption = " ".join(parts) + "."
    return caption


## Protein pre processing (binarization)

In [7]:
adt_counts = protein.X
if issparse(adt_counts):
    adt_counts = adt_counts.toarray()

# CLR normalization
adt_clr = np.log1p(adt_counts / (np.exp(np.mean(np.log1p(adt_counts), axis=1, keepdims=True))))

# Binarize above per-protein median
adt_thresholds = np.median(adt_clr, axis=0)
adt_binary = (adt_clr > adt_thresholds).astype(int)

cell_types = rna.obs["CellType"].values

print(f"ADT binary: {adt_binary.shape}")
print(f"Mean expression rate: {adt_binary.mean():.2f}")

# Test a caption
print("\nExample captions:")
for i in [0, 1000, 5000]:
    print(f"  {generate_caption(adt_binary[i], protein_names, cell_types[i])}")


ADT binary: (180794, 273)
Mean expression rate: 0.35

Example captions:
  This gdT expresses TCRVdelta2, CD161 does not express TIM, CD41, CD49b, CD14, CD137, IgD, CD86, CD70, CD103.
  This B naive expresses CD140b does not express CD103, CD244, CD226, CD328, CD150.
  This CD8 TEM does not express CD155, HLA-DR_DP_DQ, CD314, CD13, CD32, CD137.


## c) Splitting the dataset

In [8]:
run_sizes = rna.obs["Run"].value_counts()
print(f"Run sizes:\n{run_sizes}\n")

# Hold out smallest run for test, second smallest for val
test_run = run_sizes.index[-1]
val_run  = run_sizes.index[-2]
runs = rna.obs["Run"].unique()

train_mask = rna.obs["Run"].isin([r for r in runs if r not in [test_run, val_run]])
val_mask   = rna.obs["Run"] == val_run
test_mask  = rna.obs["Run"] == test_run

train_idx = np.where(train_mask.values)[0]
val_idx   = np.where(val_mask.values)[0]
test_idx  = np.where(test_mask.values)[0]

print(f"Train: {len(train_idx):>6}  ({len(train_idx)/len(rna)*100:.0f}%)")
print(f"Val:   {len(val_idx):>6}  ({len(val_idx)/len(rna)*100:.0f}%)")
print(f"Test:  {len(test_idx):>6}  ({len(test_idx)/len(rna)*100:.0f}%)")
print(f"\nTest run:  {test_run}")
print(f"Val run:   {val_run}")

# Held-out proteins for zero-shot evaluation
HOLDOUT_PROTEINS = ["CD3", "CD19", "CD56", "CD14", "CD8", "HLA-DR"]
print(f"\nHeld-out proteins: {HOLDOUT_PROTEINS}")


Run sizes:
Run
PBMC-06    45013
PBMC-07    35766
PBMC-04    34172
PBMC-05    27313
PBMC-03    23845
PBMC-02    14685
Name: count, dtype: int64

Train: 142264  (79%)
Val:    23845  (13%)
Test:   14685  (8%)

Test run:  PBMC-02
Val run:   PBMC-03

Held-out proteins: ['CD3', 'CD19', 'CD56', 'CD14', 'CD8', 'HLA-DR']


## d) Use LLM to create caption embeddings and cache them

In [9]:
import os

biobert_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
tokenizer = AutoTokenizer.from_pretrained(biobert_name)
biobert = AutoModel.from_pretrained(biobert_name).to(device)
biobert.eval()
for p in biobert.parameters():
    p.requires_grad = False

D_PROTEIN_ENC = biobert.config.hidden_size  # 768
print(f"Loaded: {biobert_name}, dim={D_PROTEIN_ENC}")

CACHE_RNA  = "/home/lemanic-g04/cache/gse_rna_hvg_encodings.npy"  # new cache name!
CACHE_PROT = "/home/lemanic-g04/cache/gse_protein_encodings_cache.npy"

# RNA encodings — save/load
if os.path.exists(CACHE_RNA):
    rna_encodings = np.load(CACHE_RNA)
    print(f"Loaded RNA encodings from cache: {rna_encodings.shape}")
else:
    np.save(CACHE_RNA, rna_encodings)
    print(f"Saved RNA encodings: {rna_encodings.shape}")

# Caption cache — save/load
N_CAPTIONS_PER_CELL = 5
BATCH_SIZE_ENCODE   = 256
n_cells = rna.shape[0]

if os.path.exists(CACHE_PROT):
    protein_encodings_cache = np.load(CACHE_PROT)
    print(f"Loaded protein cache from disk: {protein_encodings_cache.shape}")
else:
    protein_encodings_cache = np.zeros((n_cells, N_CAPTIONS_PER_CELL, D_PROTEIN_ENC), dtype=np.float32)
    print(f"Caching {N_CAPTIONS_PER_CELL} captions x {n_cells} cells...")

    for cap_idx in range(N_CAPTIONS_PER_CELL):
        all_captions = [generate_caption(adt_binary[i], protein_names, cell_types[i]) for i in range(n_cells)]
        for start in range(0, n_cells, BATCH_SIZE_ENCODE):
            end = min(start + BATCH_SIZE_ENCODE, n_cells)
            tokens = tokenizer(all_captions[start:end], padding=True, truncation=True,
                               max_length=128, return_tensors="pt")
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                emb = biobert(**tokens).last_hidden_state[:, 0, :].cpu().numpy()
            protein_encodings_cache[start:end, cap_idx] = emb
        print(f"  Caption variant {cap_idx+1}/{N_CAPTIONS_PER_CELL} done")

    np.save(CACHE_PROT, protein_encodings_cache)
    print(f"Saved protein cache: {protein_encodings_cache.shape}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 32587.61it/s]
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext, dim=768
Saved RNA encodings: (180794, 2000)
Loaded protein cache from disk: (180794, 5, 768)


## e) Dataset and dataloader

In [10]:
class CITEseqCLIPDataset(Dataset):
    def __init__(self, cell_indices, rna_encodings, protein_encodings_cache):
        self.cell_indices = cell_indices
        self.rna_encodings = rna_encodings
        self.protein_encodings_cache = protein_encodings_cache
        self.n_captions = protein_encodings_cache.shape[1]
    
    def __len__(self):
        return len(self.cell_indices)
    
    def __getitem__(self, idx):
        cell_idx = self.cell_indices[idx]
        rna_enc = torch.tensor(self.rna_encodings[cell_idx], dtype=torch.float32)
        cap_idx = random.randint(0, self.n_captions - 1)
        prot_enc = torch.tensor(self.protein_encodings_cache[cell_idx, cap_idx], dtype=torch.float32)
        return rna_enc, prot_enc

BATCH_SIZE = 512

train_dataset = CITEseqCLIPDataset(train_idx, rna_encodings, protein_encodings_cache)
val_dataset   = CITEseqCLIPDataset(val_idx,   rna_encodings, protein_encodings_cache)
test_dataset  = CITEseqCLIPDataset(test_idx,  rna_encodings, protein_encodings_cache)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


Train batches: 277, Val batches: 47, Test batches: 29


In [11]:
class RNAEncoder(nn.Module):
    def __init__(self, d_in, d_hidden, d_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_hidden, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_hidden, d_emb),
        )
    def forward(self, x):
        return self.net(x)

class ProjectionHead(nn.Module):
    def __init__(self, d_in, d_hidden, d_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, d_emb),
        )
    def forward(self, x):
        return self.net(x)

class CLIPCITE(nn.Module):
    def __init__(self, d_rna, d_protein, d_hidden=512, d_emb=256):
        super().__init__()
        self.rna_projection     = RNAEncoder(d_rna, d_hidden, d_emb)
        self.protein_projection = ProjectionHead(d_protein, d_hidden, d_emb)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
    
    def forward(self, rna_enc, protein_enc):
        rna_emb     = F.normalize(self.rna_projection(rna_enc),         dim=-1)
        protein_emb = F.normalize(self.protein_projection(protein_enc), dim=-1)
        logit_scale = self.logit_scale.exp().clamp(max=100)
        logits      = logit_scale * rna_emb @ protein_emb.T
        return logits, rna_emb, protein_emb

    def get_rna_embedding(self, rna_enc):
        return F.normalize(self.rna_projection(rna_enc), dim=-1)
    
    def get_protein_embedding(self, protein_enc):
        return F.normalize(self.protein_projection(protein_enc), dim=-1)

def clip_loss(logits):
    n      = logits.shape[0]
    labels = torch.arange(n, device=logits.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

# D_RNA_ENC is now 2000, set in the preprocessing cell
D_HIDDEN  = 512
D_EMB     = 256

model = CLIPCITE(D_RNA_ENC, D_PROTEIN_ENC, D_HIDDEN, D_EMB).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(model)


Model parameters: 1,945,601
CLIPCITE(
  (rna_projection): RNAEncoder(
    (net): Sequential(
      (0): Linear(in_features=2000, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=512, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (6): GELU(approximate='none')
      (7): Dropout(p=0.1, inplace=False)
      (8): Linear(in_features=512, out_features=256, bias=True)
    )
  )
  (protein_projection): ProjectionHead(
    (net): Sequential(
      (0): Linear(in_features=768, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=256, bias=True)
    )
  )
)


## f) Model and Training loop

In [12]:
def compute_retrieval_accuracy(logits):
    n      = logits.shape[0]
    labels = torch.arange(n, device=logits.device)
    rna_to_prot = (logits.argmax(dim=1) == labels).float().mean().item()
    prot_to_rna = (logits.argmax(dim=0) == labels).float().mean().item()
    return (rna_to_prot + prot_to_rna) / 2

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, total_acc, n_batches = 0, 0, 0
    for rna_enc, prot_enc in loader:
        rna_enc, prot_enc = rna_enc.to(device), prot_enc.to(device)
        logits, _, _ = model(rna_enc, prot_enc)
        total_loss += clip_loss(logits).item()
        total_acc  += compute_retrieval_accuracy(logits)
        n_batches  += 1
    return total_loss / n_batches, total_acc / n_batches

N_EPOCHS     = 30
LR           = 3e-4
WEIGHT_DECAY = 1e-4
MODEL_PATH   = "/home/lemanic-g04/cache/clip_cite_v2_best.pt"

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_loss = float("inf")

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss, epoch_acc, n_batches = 0, 0, 0
    
    for rna_enc, prot_enc in train_loader:
        rna_enc, prot_enc = rna_enc.to(device), prot_enc.to(device)
        logits, _, _ = model(rna_enc, prot_enc)
        loss = clip_loss(logits)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc  += compute_retrieval_accuracy(logits)
        n_batches  += 1
    
    scheduler.step()
    train_loss = epoch_loss / n_batches
    train_acc  = epoch_acc  / n_batches
    val_loss, val_acc = evaluate(model, val_loader, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{N_EPOCHS} | "
              f"Train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
              f"Val loss: {val_loss:.4f}, acc: {val_acc:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.2e}")

print(f"\nBest val loss: {best_val_loss:.4f}")


Epoch   1/30 | Train loss: 4.2726, acc: 0.0422 | Val loss: 4.1419, acc: 0.0345 | LR: 2.99e-04
Epoch   5/30 | Train loss: 3.6938, acc: 0.0676 | Val loss: 4.0870, acc: 0.0374 | LR: 2.80e-04
Epoch  10/30 | Train loss: 3.4762, acc: 0.0772 | Val loss: 4.0974, acc: 0.0390 | LR: 2.25e-04
Epoch  15/30 | Train loss: 3.3575, acc: 0.0849 | Val loss: 4.1602, acc: 0.0368 | LR: 1.50e-04
Epoch  20/30 | Train loss: 3.3033, acc: 0.0898 | Val loss: 4.2021, acc: 0.0371 | LR: 7.50e-05
Epoch  25/30 | Train loss: 3.2738, acc: 0.0948 | Val loss: 4.2239, acc: 0.0372 | LR: 2.01e-05
Epoch  30/30 | Train loss: 3.2630, acc: 0.0980 | Val loss: 4.2310, acc: 0.0377 | LR: 0.00e+00

Best val loss: 4.0610
